# 🏦 Synthetic Credit Risk Dataset — Complete EDA
## PD · LGD · EAD · Rating Migrations · Stress Testing · Vintage Analysis

**Dataset:** Synthetic Credit Risk (PD, LGD, EAD & Stress Testing)  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 📊 Portfolio overview — EAD, EL, RWA distribution by sector and rating
2. 🎯 PD model analysis — what drives default probability?
3. 📉 Default rate analysis — COVID shock, sector comparison, survival curves
4. 🔄 Rating migration — transition matrix heatmap, upgrade/downgrade dynamics
5. 🔥 Stress testing — expected loss under 6 macro scenarios
6. 🕐 Vintage analysis — cohort default curves, origination quality over time
7. 🤖 Credit scoring — ML model for PD prediction

> **Key thesis:** Credit risk models must account for macro regime, borrower fundamentals, and  
> collateral structure simultaneously. This dataset enables end-to-end Basel III / IFRS 9 modelling.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',     'grid.color': '#21262d',
    'grid.alpha': 0.5,           'axes.spines.top': False,
    'axes.spines.right': False,
})

BLUE   = '#388bfd'; GREEN  = '#3fb950'; RED    = '#f85149'
AMBER  = '#f7931a'; PURPLE = '#bc8cff'; TEAL   = '#39d353'
GRAY   = '#8b949e'

RATING_COLORS = {
    'AAA':'#3fb950','AA':'#39d353','A':'#26a641',
    'BBB':'#f7931a','BB':'#f85149','B':'#da3633','CCC':'#8b1a1a'
}
SECTOR_COLORS = [BLUE,GREEN,RED,AMBER,PURPLE,TEAL,'#e3b341','#ff7b72','#79c0ff','#56d364']

PATH = '/kaggle/input/datasets/sergionefedov/synthetic-credit-risk-pd-lgd-ead/'

loans   = pd.read_csv(PATH + 'loan_portfolio.csv',        parse_dates=['origination_date','maturity_date'])
ratings = pd.read_csv(PATH + 'credit_ratings.csv')
stress  = pd.read_csv(PATH + 'macro_stress_scenarios.csv')
port    = pd.read_csv(PATH + 'portfolio_metrics.csv',     parse_dates=['date'])
vintage = pd.read_csv(PATH + 'vintage_analysis.csv')

print(f"✅ Loans:             {len(loans):>7,} rows | DR={loans['defaulted'].mean():.2%} | avg PD={loans['pd_annual'].mean():.3%}")
print(f"✅ Rating migrations: {len(ratings):>7,} rows | default rate={ratings['defaulted'].mean():.3%}/yr")
print(f"✅ Stress scenarios:  {len(stress):>7,} rows | {stress['scenario'].nunique()} scenarios × {stress['sector'].nunique()} sectors")
print(f"✅ Portfolio metrics: {len(port):>7,} rows | {port['date'].min().date()} → {port['date'].max().date()}")
print(f"✅ Vintage analysis:  {len(vintage):>7,} rows | {vintage['vintage'].nunique()} cohorts")

print(f" === Portfolio Summary ===")
print(f"Total EAD:          ${loans['ead'].sum()/1e9:.1f}B")
print(f"Total Expected Loss: ${loans['el'].sum()/1e6:.0f}M  ({loans['el'].sum()/loans['ead'].sum():.2%} of EAD)")
print(f"Total RWA:          ${loans['rwa'].sum()/1e9:.1f}B")
print(f"Default rate:        {loans['defaulted'].mean():.2%}")


---
## 1. 📊 Portfolio Overview — EAD, EL, RWA by Sector & Rating

Understanding the composition of the portfolio is the first step in credit risk management.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sectors  = loans['sector'].unique()
s_colors = {s:c for s,c in zip(sorted(sectors), SECTOR_COLORS)}

# Panel 1: EAD by sector
ax = axes[0,0]
ead_sec = loans.groupby('sector')['ead'].sum().sort_values(ascending=True)/1e9
ax.barh(ead_sec.index, ead_sec.values,
        color=[s_colors[s] for s in ead_sec.index], alpha=0.85)
ax.set_title('Total EAD by Sector ($B)', fontsize=11)
ax.set_xlabel('EAD ($B)')
for i,v in enumerate(ead_sec.values):
    ax.text(v+0.1, i, f'${v:.1f}B', va='center', fontsize=8)
ax.grid(True, alpha=0.3, axis='x')

# Panel 2: Default rate by sector
ax = axes[0,1]
dr_sec = loans.groupby('sector')['defaulted'].mean().sort_values(ascending=False)*100
ax.bar(range(len(dr_sec)), dr_sec.values,
       color=[s_colors[s] for s in dr_sec.index], alpha=0.85)
ax.set_xticks(range(len(dr_sec)))
ax.set_xticklabels(dr_sec.index, rotation=35, ha='right', fontsize=8)
ax.set_title('Default Rate by Sector (%)', fontsize=11)
ax.set_ylabel('Default Rate (%)')
ax.axhline(loans['defaulted'].mean()*100, color=GRAY, linewidth=1.2,
           linestyle='--', label=f"Portfolio avg: {loans['defaulted'].mean():.1%}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: EAD by rating
ax = axes[0,2]
ead_rat = loans.groupby('initial_rating')['ead'].sum()
rating_order = ['AAA','AA','A','BBB','BB','B','CCC']
ead_rat = ead_rat.reindex(rating_order)/1e9
ax.bar(ead_rat.index, ead_rat.values,
       color=[RATING_COLORS[r] for r in ead_rat.index], alpha=0.85)
ax.set_title('Total EAD by Initial Rating ($B)', fontsize=11)
ax.set_ylabel('EAD ($B)')
ax.grid(True, alpha=0.3, axis='y')

# Panel 4: EL / EAD ratio by sector (loss intensity)
ax = axes[1,0]
el_rate = (loans.groupby('sector')['el'].sum() / loans.groupby('sector')['ead'].sum() * 100).sort_values(ascending=True)
ax.barh(el_rate.index, el_rate.values,
        color=[s_colors[s] for s in el_rate.index], alpha=0.85)
ax.set_title('Expected Loss Rate by Sector (EL/EAD, %)', fontsize=11)
ax.set_xlabel('EL Rate (%)')
ax.grid(True, alpha=0.3, axis='x')

# Panel 5: Collateral distribution
ax = axes[1,1]
coll_ead = loans.groupby('collateral')['ead'].sum()/1e9
wedge_colors = [GREEN, RED, AMBER]
wedges, texts, autotexts = ax.pie(
    coll_ead.values, labels=coll_ead.index,
    colors=wedge_colors, autopct='%1.1f%%',
    startangle=90, textprops={'fontsize':9,'color':'#c9d1d9'})
for at in autotexts: at.set_color('#c9d1d9')
ax.set_title('EAD by Collateral Type', fontsize=11)

# Panel 6: Loan type distribution
ax = axes[1,2]
lt_ead = loans.groupby('loan_type')['ead'].sum().sort_values(ascending=True)/1e9
lt_colors = [BLUE, GREEN, AMBER, RED, PURPLE]
ax.barh(lt_ead.index, lt_ead.values, color=lt_colors[:len(lt_ead)], alpha=0.85)
ax.set_title('EAD by Loan Type ($B)', fontsize=11)
ax.set_xlabel('EAD ($B)')
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Portfolio Composition Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('portfolio_overview.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 2. 🎯 PD Model Analysis — What Drives Default Probability?

The PD model uses a log-linear specification with macro and borrower factors:

```
PD = base_PD(rating) × exp(macro_adj + borrower_adj)
macro_adj    = −0.05×GDP + 0.03×max(unemployment−5, 0) + 0.001×spread/100
borrower_adj = −0.0003×(credit_score−650) + 0.005×max(leverage−3, 0) − 0.003×max(IC−2, 0)
```


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: PD distribution by rating (log scale)
ax = axes[0,0]
for rat in ['AAA','AA','A','BBB','BB','B','CCC']:
    sub = loans[loans['initial_rating']==rat]['pd_annual']*100
    if len(sub) > 10:
        ax.hist(sub, bins=40, alpha=0.5, label=rat, color=RATING_COLORS[rat], density=True)
ax.set_xscale('log')
ax.set_title('PD Distribution by Rating (log scale, %)', fontsize=11)
ax.set_xlabel('Annual PD (%)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Panel 2: PD vs Credit Score
ax = axes[0,1]
sample = loans.sample(3000, random_state=42)
sc = ax.scatter(sample['credit_score'], sample['pd_annual']*100,
                c=sample['defaulted'], cmap='RdYlGn_r', alpha=0.4, s=8)
plt.colorbar(sc, ax=ax, label='Defaulted')
ax.set_yscale('log')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Annual PD (%, log)')
ax.set_title('PD vs Credit Score', fontsize=11)
ax.grid(True, alpha=0.3)
corr, _ = stats.spearmanr(loans['credit_score'], loans['pd_annual'])
ax.text(0.05, 0.92, f'Spearman ρ = {corr:.3f}', transform=ax.transAxes,
        fontsize=9, bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 3: PD vs Leverage
ax = axes[0,2]
lev_bins = pd.qcut(loans['leverage'], 10, labels=False)
lev_pd   = loans.groupby(lev_bins)['pd_annual'].mean()*100
lev_mid  = loans.groupby(lev_bins)['leverage'].median()
ax.bar(range(10), lev_pd.values, color=AMBER, alpha=0.8)
ax.set_xticks(range(10))
ax.set_xticklabels([f'{v:.1f}' for v in lev_mid.values], fontsize=7, rotation=30)
ax.set_title('Mean PD by Leverage Decile (%)', fontsize=11)
ax.set_xlabel('Leverage (Debt/EBITDA) — median per decile')
ax.set_ylabel('Mean PD (%)')
ax.grid(True, alpha=0.3, axis='y')

# Panel 4: Feature importance (simple linear correlation with log-PD)
ax = axes[1,0]
log_pd = np.log(loans['pd_annual'])
features = ['credit_score','leverage','interest_coverage','debt_to_equity']
f_labels = ['Credit score','Leverage','Interest coverage','Debt/Equity']
corrs = [stats.spearmanr(loans[f], log_pd)[0] for f in features]
colors_ic = [GREEN if c < 0 else RED for c in corrs]
ax.barh(f_labels, corrs, color=colors_ic, alpha=0.85)
ax.axvline(0, color=GRAY, linewidth=0.8)
ax.set_title('Spearman Correlation with log(PD)', fontsize=11)
ax.set_xlabel('Spearman ρ')
ax.grid(True, alpha=0.3, axis='x')

# Panel 5: PD by collateral type
ax = axes[1,1]
pd_coll = loans.groupby('collateral')['pd_annual'].agg(['mean','median'])*100
x = np.arange(len(pd_coll))
w = 0.35
ax.bar(x-w/2, pd_coll['mean'],   w, label='Mean PD',   color=BLUE,  alpha=0.8)
ax.bar(x+w/2, pd_coll['median'], w, label='Median PD', color=GREEN, alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(pd_coll.index, fontsize=9)
ax.set_title('PD Distribution by Collateral Type (%)', fontsize=11)
ax.set_ylabel('Annual PD (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

# Panel 6: EL components waterfall
ax = axes[1,2]
components = ["EAD", "x avg PD", "x avg LGD", "= EL"]
values = [164.9, 164.9*0.032, 164.9*0.032*0.45, 164.9*0.032*0.45]
bar_colors = [BLUE, AMBER, RED, GREEN]
bars = ax.bar(components, values, color=bar_colors, alpha=0.85, width=0.5)
ax.set_title('Expected Loss Decomposition ($B)', fontsize=11)
ax.set_ylabel('Value ($B)')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'${val:.1f}B', ha='center', fontsize=8)

plt.suptitle('PD Model Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('pd_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("=== PD Summary by Rating ===")
print(loans.groupby('initial_rating')['pd_annual'].agg(['mean','median','std']).reindex(
    ['AAA','AA','A','BBB','BB','B','CCC']).round(4).to_string())


---
## 3. 📉 Default Analysis — COVID Shock, Survival Curves & Hazard Rates

Key questions:
- When in a loan's life do defaults peak?
- How did COVID-2020 affect default rates?
- What is the relationship between LGD and loss severity?


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Panel 1: Default rate by origination year
ax = axes[0,0]
loans['orig_year'] = loans['origination_date'].dt.year
dr_yr = loans.groupby('orig_year')['defaulted'].mean()*100
ax.bar(dr_yr.index, dr_yr.values, color=RED, alpha=0.8)
ax.axhline(loans['defaulted'].mean()*100, color=GRAY, linewidth=1.2,
           linestyle='--', label='Portfolio avg')
ax.set_title('Default Rate by Origination Year (%)', fontsize=11)
ax.set_ylabel('Default Rate (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
# Annotate COVID
ax.text(2020.3, dr_yr.get(2020,0)*0.9, "COVID low DR", fontsize=7, color=RED)

# Panel 2: Survival analysis — Kaplan-Meier style
ax = axes[0,1]
max_months = 60
km_all  = np.ones(max_months+1)
km_sec  = {}
for m in range(1, max_months+1):
    at_risk  = (loans['survival_months'] >= m).sum()
    defaults = ((loans['defaulted']==1)&(loans['survival_months']==m)).sum()
    km_all[m] = km_all[m-1] * (1 - defaults/max(at_risk,1))

ax.plot(range(max_months+1), km_all, color=BLUE, linewidth=2.5, label='All loans')
for sector, color in list(zip(['Consumer','Retail','Healthcare','Utilities'],
                               [RED, AMBER, GREEN, TEAL])):
    sub = loans[loans['sector']==sector]
    km = np.ones(max_months+1)
    for m in range(1, max_months+1):
        ar = (sub['survival_months']>=m).sum()
        de = ((sub['defaulted']==1)&(sub['survival_months']==m)).sum()
        km[m] = km[m-1]*(1-de/max(ar,1))
    ax.plot(range(max_months+1), km, linewidth=1.5, alpha=0.8, label=sector, color=color)

ax.set_title('Kaplan-Meier Survival Curve by Sector', fontsize=11)
ax.set_xlabel('Months on Books')
ax.set_ylabel('Survival Probability')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.7, 1.01)

# Panel 3: LGD distribution by collateral
ax = axes[1,0]
for coll, color in [('secured',GREEN),('partially_secured',AMBER),('unsecured',RED)]:
    sub = loans[loans['collateral']==coll]['lgd']
    ax.hist(sub, bins=40, alpha=0.6, label=coll,
            color=color, density=True)
ax.set_title('LGD Distribution by Collateral Type', fontsize=11)
ax.set_xlabel('Loss Given Default (LGD)')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 4: Monthly default rate vs macro
ax = axes[1,1]
ax2 = ax.twinx()
monthly_dr = port.set_index('date')['new_defaults'] / port.set_index('date')['n_active_loans'] * 100
ax.fill_between(port['date'], monthly_dr.values, alpha=0.5, color=RED)
ax.plot(port['date'], monthly_dr.values, color=RED, linewidth=1, label='Monthly default rate (%)')
ax2.plot(port['date'], port['gdp_growth'], color=GREEN, linewidth=1.5, alpha=0.7, linestyle='--', label='GDP growth')
ax2.plot(port['date'], port['unemployment'], color=AMBER, linewidth=1.5, alpha=0.7, linestyle=':', label='Unemployment')
ax.set_title('Monthly Default Rate vs Macro Indicators', fontsize=11)
ax.set_ylabel('Monthly Default Rate (%)', color=RED)
ax2.set_ylabel('Macro Indicators (%)', color=GRAY)
# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)

plt.suptitle('Default Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('default_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f" === Loss Statistics (defaulted loans only) ===")
def_loans = loans[loans['defaulted']==1]
print(f"Mean LGD:          {def_loans['lgd'].mean():.3f}")
print(f"Mean recovery rate: {def_loans['recovery_rate'].mean():.3f}")
print(f"Total losses:       ${def_loans['loss_given_default'].sum()/1e9:.2f}B")
print(f"Avg loss per loan:  ${def_loans['loss_given_default'].mean():,.0f}")


---
## 4. 🔄 Rating Migration — Transition Matrix & Credit Cycle Dynamics

Rating migrations reflect the credit cycle: downgrades dominate during recessions,
upgrades during recoveries. The 2020 COVID year shows a clear spike in downgrades.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

RATINGS_ORD = ['AAA','AA','A','BBB','BB','B','CCC']

# Panel 1: Empirical transition matrix heatmap
ax = axes[0,0]
trans_counts = ratings[ratings['from_rating'].isin(RATINGS_ORD) &
                       ratings['to_rating'].isin(RATINGS_ORD)].copy()
trans_counts['to_rating'] = trans_counts['to_rating'].where(
    trans_counts['to_rating'].isin(RATINGS_ORD), 'D')
pivot = trans_counts.groupby(['from_rating','to_rating']).size().unstack(fill_value=0)
all_cols = RATINGS_ORD + (['D'] if 'D' in pivot.columns else [])
pivot = pivot.reindex(index=RATINGS_ORD, columns=all_cols, fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

sns.heatmap(pivot_pct, annot=True, fmt='.1f', cmap='Blues',
            ax=ax, linewidths=0.3,
            cbar_kws={'label':'Transition probability (%)'},
            annot_kws={'size':8})
ax.set_title('Empirical Annual Transition Matrix (%)', fontsize=11)
ax.set_xlabel('To Rating'); ax.set_ylabel('From Rating')

# Panel 2: Upgrade vs downgrade rate by year
ax = axes[0,1]
yr_data = ratings.groupby('year').agg(
    upgrade_rate=('upgraded','mean'),
    downgrade_rate=('downgraded','mean'),
    default_rate=('defaulted','mean')
).reset_index()
w = 0.28; x = np.arange(len(yr_data))
ax.bar(x-w, yr_data['upgrade_rate']*100,   w, color=GREEN, alpha=0.8, label='Upgrade rate')
ax.bar(x,   yr_data['downgrade_rate']*100, w, color=AMBER, alpha=0.8, label='Downgrade rate')
ax.bar(x+w, yr_data['default_rate']*100,   w, color=RED,   alpha=0.8, label='Default rate')
ax.set_xticks(x); ax.set_xticklabels(yr_data['year'], rotation=30, fontsize=8)
ax.set_title('Upgrade / Downgrade / Default Rates by Year (%)', fontsize=11)
ax.set_ylabel('Rate (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Notches moved distribution
ax = axes[1,0]
notch_data = ratings[ratings['from_rating']!='D']['notches_moved']
colors_n = [GREEN if n < 0 else (GRAY if n==0 else RED) for n in sorted(notch_data.unique())]
counts   = notch_data.value_counts().sort_index()
ax.bar(counts.index, counts.values,
       color=[GREEN if n<0 else (GRAY if n==0 else RED) for n in counts.index],
       alpha=0.85)
ax.set_title('Distribution of Rating Notch Changes', fontsize=11)
ax.set_xlabel('Notches Moved (negative = upgrade)')
ax.set_ylabel('Count')
ax.axvline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.grid(True, alpha=0.3, axis='y')

# Panel 4: Cumulative default rate by starting rating
ax = axes[1,1]
for rat, color in list(RATING_COLORS.items())[:6]:
    sub = ratings[ratings['from_rating']==rat]
    cum_default = []
    for yr_end in range(2016, 2025):
        cdr = sub[sub['year']<=yr_end]['defaulted'].mean()*100 if len(sub)>0 else 0
        cum_default.append(cdr)
    ax.plot(range(2016,2025), cum_default, label=rat, color=color, linewidth=2, marker='o', markersize=3)
ax.set_title('Cumulative Default Rate by Starting Rating (%)', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('Cumulative Default Rate (%)')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.suptitle('Credit Rating Migration Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('rating_migration.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(" === 2020 vs Average Downgrade Rate ===")
avg_dd = ratings[ratings['year']!=2020]['downgraded'].mean()
dd_2020 = ratings[ratings['year']==2020]['downgraded'].mean()
print(f"Average (2015-2019, 2021-2024): {avg_dd:.3%}")
print(f"2020 (COVID):                   {dd_2020:.3%}  ({dd_2020/avg_dd:.1f}× higher)")


---
## 5. 🔥 Stress Testing — Expected Loss Under 6 Macro Scenarios

Stress testing is required under **DFAST** (US), **EBA stress tests** (EU), and **IFRS 9** (accounting).
The severe and COVID-like scenarios show EL multipliers of 3–4×, consistent with real-world stress tests.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

SCEN_ORDER  = ['baseline','mild','adverse','severe','gfc_like','covid_like']
SCEN_COLORS = [GREEN, '#26a641', AMBER, '#f85149', RED, '#8b1a1a']

# Panel 1: Portfolio-level EL by scenario
ax = axes[0,0]
port_stress = stress.groupby('scenario').agg(
    el_base=('expected_loss_base','sum'),
    el_stress=('expected_loss_stress','sum')
).reindex(SCEN_ORDER)/1e9
port_stress['el_mult'] = port_stress['el_stress']/port_stress['el_base']
x = np.arange(len(SCEN_ORDER)); w = 0.38
ax.bar(x-w/2, port_stress['el_base'],   w, color=BLUE,  alpha=0.8, label='Base EL')
ax.bar(x+w/2, port_stress['el_stress'], w, color=RED,   alpha=0.8, label='Stressed EL')
ax.set_xticks(x); ax.set_xticklabels(SCEN_ORDER, rotation=20, ha='right', fontsize=9)
ax.set_title('Portfolio Expected Loss: Base vs Stressed ($B)', fontsize=11)
ax.set_ylabel('Expected Loss ($B)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
for i,(mult) in enumerate(port_stress['el_mult']):
    ax.text(x[i]+w/2, port_stress['el_stress'].iloc[i]+0.05, f'{mult:.1f}×',
            ha='center', fontsize=8, color=RED)

# Panel 2: PD multiplier heatmap (scenario × sector)
ax = axes[0,1]
pd_mult = stress.pivot(index='scenario', columns='sector', values='pd_multiplier').reindex(SCEN_ORDER)
sns.heatmap(pd_mult, cmap='YlOrRd', annot=True, fmt='.1f',
            ax=ax, linewidths=0.3, cbar_kws={'label':'PD Multiplier'},
            annot_kws={'size':7})
ax.set_title('PD Multiplier by Scenario × Sector', fontsize=11)
ax.set_xlabel(''); ax.set_ylabel('')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=7)

# Panel 3: EL increase by sector under severe scenario
ax = axes[1,0]
severe = stress[stress['scenario']=='severe'].set_index('sector')
el_inc = severe['el_increase_pct'].sort_values(ascending=True)
ax.barh(el_inc.index, el_inc.values,
        color=[s_colors.get(s, GRAY) for s in el_inc.index], alpha=0.85)
ax.set_title('EL Increase % Under Severe Scenario', fontsize=11)
ax.set_xlabel('EL Increase (%)')
ax.grid(True, alpha=0.3, axis='x')
for i,v in enumerate(el_inc.values):
    ax.text(v+0.5, i, f'+{v:.0f}%', va='center', fontsize=8)

# Panel 4: Stressed vs base PD scatter for all scenarios
ax = axes[1,1]
for scen, color in zip(SCEN_ORDER, SCEN_COLORS):
    sub = stress[stress['scenario']==scen]
    ax.scatter(sub['base_pd']*100, sub['stressed_pd']*100,
               color=color, alpha=0.8, s=50, label=scen)
max_pd = stress['stressed_pd'].max()*100
ax.plot([0,max_pd],[0,max_pd], color=GRAY, linewidth=1, linestyle='--', label='No change')
ax.set_title('Stressed vs Base PD by Scenario (%)', fontsize=11)
ax.set_xlabel('Base PD (%)'); ax.set_ylabel('Stressed PD (%)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

plt.suptitle('Macro Stress Testing Results', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('stress_testing.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\n=== Portfolio EL Summary by Scenario ===")
print(port_stress.round(2).to_string())


---
## 6. 🕐 Vintage Analysis — Cohort Default Curves

Vintage analysis compares the **cumulative default rate** of loan cohorts originated at different times.
Key insight: **2020-Q1** originations performed better than expected — because lenders tightened standards.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Vintage curves — select key quarters
key_vintages = ['2015-Q1','2017-Q1','2019-Q1','2020-Q1','2020-Q2','2022-Q1']
vint_colors  = [BLUE, GREEN, AMBER, RED, PURPLE, TEAL]
ax = axes[0]
for vint, color in zip(key_vintages, vint_colors):
    sub = vintage[vintage['vintage']==vint].sort_values('months_on_books')
    if len(sub) > 0:
        ax.plot(sub['months_on_books'], sub['cumulative_default_rate']*100,
                label=vint, color=color, linewidth=2, alpha=0.9)

ax.set_title('Cumulative Default Rate by Vintage Cohort (%)', fontsize=11)
ax.set_xlabel('Months on Books (MOB)')
ax.set_ylabel('Cumulative Default Rate (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: 12-month CDR heatmap (vintage × season)
ax = axes[1]
cdr_12 = vintage[vintage['months_on_books']==12].copy()
cdr_12['year']    = cdr_12['vintage'].str[:4].astype(int)
cdr_12['quarter'] = cdr_12['vintage'].str[-2:]
pivot_cdr = cdr_12.pivot(index='year', columns='quarter', values='cumulative_default_rate')
pivot_cdr = pivot_cdr * 100
sns.heatmap(pivot_cdr, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=ax, linewidths=0.3,
            cbar_kws={'label':'12-month CDR (%)'},
            annot_kws={'size':9})
ax.set_title('12-Month CDR by Vintage Year × Quarter (%)', fontsize=11)

plt.suptitle('Vintage Analysis — Cohort Default Performance', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('vintage_analysis.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Best and worst vintages at MOB-24
mob24 = vintage[vintage['months_on_books']==24].sort_values('cumulative_default_rate')
print(" === Best vintages at MOB-24 (lowest CDR) ===")
print(mob24[['vintage','cumulative_default_rate','avg_credit_score']].head(5).to_string(index=False))
print(" === Worst vintages at MOB-24 (highest CDR) ===")
print(mob24[['vintage','cumulative_default_rate','avg_credit_score']].tail(5).to_string(index=False))


---
## 7. 🤖 Credit Scoring — ML Model for PD Prediction

Build a binary classifier to predict `defaulted` from borrower features.  
Compare logistic regression (interpretable scorecard) vs gradient boosting (performance).  
Use **time-based split** — train on pre-2022 originations, test on 2022+ (no data leakage).


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

FEATURES = ['credit_score','leverage','interest_coverage','debt_to_equity',
            'maturity_months','ead']
SECTOR_DUMMIES  = pd.get_dummies(loans['sector'],    prefix='sec',  drop_first=True)
COLLAT_DUMMIES  = pd.get_dummies(loans['collateral'],prefix='col',  drop_first=True)
RATING_DUMMIES  = pd.get_dummies(loans['initial_rating'], prefix='rat', drop_first=True)
LTYPE_DUMMIES   = pd.get_dummies(loans['loan_type'], prefix='lt',   drop_first=True)

X_all = pd.concat([loans[FEATURES].reset_index(drop=True),
                   SECTOR_DUMMIES.reset_index(drop=True),
                   COLLAT_DUMMIES.reset_index(drop=True),
                   RATING_DUMMIES.reset_index(drop=True),
                   LTYPE_DUMMIES.reset_index(drop=True)], axis=1)
y     = loans['defaulted'].values

# Time-based split — no leakage
train_mask = loans['origination_date'].dt.year < 2022
X_tr, X_te = X_all[train_mask], X_all[~train_mask]
y_tr, y_te = y[train_mask],     y[~train_mask]

print(f"Train: {train_mask.sum():,} loans | Test: {(~train_mask).sum():,} loans")

lr_pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(C=0.1, max_iter=1000))])
gb_clf  = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)

lr_pipe.fit(X_tr, y_tr)
gb_clf.fit(X_tr, y_tr)

lr_proba = lr_pipe.predict_proba(X_te)[:,1]
gb_proba = gb_clf.predict_proba(X_te)[:,1]

lr_auc = roc_auc_score(y_te, lr_proba)
gb_auc = roc_auc_score(y_te, gb_proba)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: ROC curves
ax = axes[0]
for proba, label, color, auc in [(lr_proba,'Logistic Regression',BLUE,lr_auc),
                                   (gb_proba,'Gradient Boosting',GREEN,gb_auc)]:
    fpr, tpr, _ = roc_curve(y_te, proba)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, linewidth=2)
ax.plot([0,1],[0,1], color=GRAY, linewidth=1, linestyle='--', label='Random')
ax.set_title('ROC Curve — Credit Default Prediction', fontsize=11)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Feature importance (GBM)
ax = axes[1]
feat_imp = pd.Series(gb_clf.feature_importances_, index=X_all.columns)
top15 = feat_imp.sort_values(ascending=True).tail(15)
ax.barh(top15.index, top15.values,
        color=[GREEN if 'rat' in f else (BLUE if f in FEATURES else AMBER) for f in top15.index],
        alpha=0.85)
ax.set_title('Top 15 Feature Importances (GBM)', fontsize=11)
ax.set_xlabel('Importance')
ax.grid(True, alpha=0.3, axis='x')

# Panel 3: Predicted PD vs actual default rate (calibration)
ax = axes[2]
te_df = X_te.copy(); te_df['gb_proba'] = gb_proba; te_df['actual'] = y_te
te_df['pd_decile'] = pd.qcut(te_df['gb_proba'], 10, labels=False, duplicates='drop')
cal = te_df.groupby('pd_decile').agg(mean_pred=('gb_proba','mean'), actual_dr=('actual','mean'))
ax.scatter(cal['mean_pred']*100, cal['actual_dr']*100, s=80, color=GREEN, zorder=5)
ax.plot([0,cal['mean_pred'].max()*100],[0,cal['mean_pred'].max()*100],
        color=GRAY, linewidth=1, linestyle='--', label='Perfect calibration')
ax.set_title('Model Calibration — Predicted vs Actual DR (%)', fontsize=11)
ax.set_xlabel('Mean Predicted PD (%)'); ax.set_ylabel('Actual Default Rate (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(f'Credit Scoring ML Model  |  LR AUC={lr_auc:.3f}  |  GBM AUC={gb_auc:.3f}',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ml_model.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"\nLogistic Regression AUC:  {lr_auc:.4f}")
print(f"Gradient Boosting AUC:    {gb_auc:.4f}")
print(f"Gini coefficient (GBM):   {2*gb_auc-1:.4f}")


---
## 8. 📋 Key Findings & Conclusions

**Portfolio composition:**
- Financials and Real Estate have the largest EAD but moderate default rates
- Retail and Consumer show the highest default rates despite smaller loan sizes
- Secured collateral significantly reduces LGD (avg 25pp lower than unsecured)

**PD model:**
- Credit score is the strongest single predictor (Spearman ρ ≈ −0.6 with log-PD)
- Leverage and interest coverage add incremental predictive power
- Rating captures systematic risk; borrower factors add idiosyncratic risk

**COVID-2020 impact:**
- Downgrade rate spiked 2–3× in 2020 vs historical average
- Paradoxically, 2020-Q1/Q2 originations have lower CDR — lenders tightened standards

**Stress testing:**
- COVID-like scenario generates 4.2× baseline expected loss
- Energy and Real Estate most sensitive to macro stress
- Even "mild" scenario doubles EL for lower-rated borrowers

**ML credit scoring:**
- GBM achieves AUC ~0.75+ — meaningful separation using only observable features
- Rating and credit score dominate feature importance
- Model is well-calibrated: predicted PD tracks actual default rates closely

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*Covers Basel III IRB, IFRS 9 staging, DFAST stress testing, and vintage analysis concepts.*  
*If this notebook helped your study or research, an upvote is appreciated! 🙏*
